In [1]:
!pip install -q transformers sentence-transformers faiss-cpu pillow pandas

In [2]:
import os, re, math
import numpy as np
import pandas as pd
import faiss
from PIL import Image
from transformers import pipeline, BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer

# 1) CLIP: classification (edible / non-edible / food type)
clip = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

# 2) BLIP: image captioning
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# 3) MiniLM: text embeddings for FAISS matching
mini = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

... (27 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded


In [19]:
from google.colab import drive
drive.mount('/content/drive')

IMAGE_DIR = "/content/drive/MyDrive/photos"
print(find_images())         # change to "/content" or a Drive folder if needed

def find_images(folder=None):
    folder = folder or IMAGE_DIR
    files = [f for f in os.listdir(folder)
             if re.fullmatch(r"f\d+\.(jpg|jpeg|png|webp)", f.lower())]
    return sorted(files, key=lambda f: int(re.findall(r"\d+", f)[0]))

def caption(path):
    img = Image.open(path).convert("RGB")
    inputs = blip_processor(img, return_tensors="pt")
    out = blip_model.generate(**inputs, max_new_tokens=30,
                              num_beams=3, repetition_penalty=1.5)
    return blip_processor.decode(out[0], skip_special_tokens=True)

def haversine(lat1, lng1, lat2, lng2):
    R = 6371
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lng2 - lng1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

print(find_images())

Mounted at /content/drive
['f2.jpeg', 'f3.jpg', 'f4.jpg', 'f5.webp', 'f6.jpg', 'f7.jpeg', 'f8.webp', 'f9.jpeg', 'f10.jpg', 'f11.jpeg', 'f12.webp', 'f13.jpg', 'f14.jpeg', 'f15.jpg']
['f2.jpeg', 'f3.jpg', 'f4.jpg', 'f5.webp', 'f6.jpg', 'f7.jpeg', 'f8.webp', 'f9.jpeg', 'f10.jpg', 'f11.jpeg', 'f12.webp', 'f13.jpg', 'f14.jpeg', 'f15.jpg']


In [20]:
IMAGE_DIR = "/content"
print(find_images())

['f1.jpg', 'f2.jpeg', 'f3.jpg', 'f4.jpg', 'f5.webp', 'f6.jpg', 'f7.jpeg', 'f8.webp', 'f9.jpeg', 'f10.jpg', 'f11.jpeg', 'f12.webp', 'f13.jpg', 'f14.jpeg', 'f15.jpg']


In [21]:
def safety_check(food_category, hours_since_cooked, storage):
    """storage: 'room', 'fridge' or 'sealed'"""
    if food_category != "cooked meals" or storage == "sealed":
        return True, "ok"
    if storage == "room" and hours_since_cooked > 4:
        return False, f"cooked food {hours_since_cooked}h at room temp (limit 4h)"
    if storage == "fridge" and hours_since_cooked > 24:
        return False, f"cooked food {hours_since_cooked}h in fridge (limit 24h)"
    return True, "ok"

print(safety_check("cooked meals", 6, "room"))
print(safety_check("cooked meals", 6, "fridge"))

(False, 'cooked food 6h at room temp (limit 4h)')
(True, 'ok')


In [23]:
# Q1: edible vs non-edible (binary)
EDIBLE_PROMPTS = ["fresh appetizing food that is safe to eat",
                  "rotten, moldy or inedible food waste"]

# Q2: if non-edible, which recovery stream
NON_EDIBLE = {
    "stale dry bread":                  "animal",
    "vegetable peels and food scraps":  "animal",
    "moldy spoiled food":               "compost",
    "rotten decaying food":             "compost",
    "bones and meat waste":             "biogas",
}

# Q3: food type -> broad category (used in the NGO query)
FOOD_LABELS = {
    "rice dish or biryani": "cooked meals", "curry or gravy": "cooked meals",
    "soup": "cooked meals", "noodles or pasta": "cooked meals",
    "pizza": "cooked meals", "chicken or meat dish": "cooked meals",
    "bread or bun": "bakery", "cake or pastry": "bakery",
    "fruits": "fruits and vegetables", "vegetables": "fruits and vegetables",
    "sweets or dessert": "sweets and desserts",
    "packaged snacks": "packaged food", "drinks": "beverages",
}

def classify_donation(path):
    img = Image.open(path).convert("RGB")

    r1 = clip(img, candidate_labels=EDIBLE_PROMPTS, hypothesis_template="a photo of {}.")
    edible_score = next(r["score"] for r in r1 if r["label"] == EDIBLE_PROMPTS[0])

    r2 = clip(img, candidate_labels=list(NON_EDIBLE), hypothesis_template="a photo of {}.")
    non_label = r2[0]["label"]

    r3 = clip(img, candidate_labels=list(FOOD_LABELS),
              hypothesis_template="a photo of {}, a type of food.")
    food_label, food_score = r3[0]["label"], r3[0]["score"]

    return {
        "edible": edible_score >= 0.5,
        "edible_score": round(edible_score, 2),
        "non_edible_label": non_label,
        "non_edible_stream": NON_EDIBLE[non_label],
        "food_type": food_label,
        "food_category": FOOD_LABELS[food_label],
        "food_score": round(food_score, 2),
        "needs_review": 0.35 < edible_score < 0.65,   # model is unsure
    }

for f in find_images():
    c = classify_donation(os.path.join(IMAGE_DIR, f))
    print(f, "| edible" if c["edible"] else "| NON-EDIBLE", c["edible_score"],
          "|", c["food_type"], "|", c["non_edible_label"])

f1.jpg | edible 0.66 | curry or gravy | rotten decaying food
f2.jpeg | edible 0.84 | pizza | rotten decaying food
f3.jpg | edible 0.96 | chicken or meat dish | rotten decaying food
f4.jpg | edible 0.98 | rice dish or biryani | rotten decaying food
f5.webp | edible 0.99 | pizza | rotten decaying food
f6.jpg | edible 0.93 | curry or gravy | rotten decaying food
f7.jpeg | edible 0.83 | noodles or pasta | rotten decaying food
f8.webp | edible 0.99 | curry or gravy | rotten decaying food
f9.jpeg | edible 0.88 | rice dish or biryani | vegetable peels and food scraps
f10.jpg | edible 0.99 | rice dish or biryani | rotten decaying food
f11.jpeg | NON-EDIBLE 0.2 | rice dish or biryani | moldy spoiled food
f12.webp | NON-EDIBLE 0.03 | sweets or dessert | bones and meat waste
f13.jpg | NON-EDIBLE 0.41 | fruits | rotten decaying food
f14.jpeg | NON-EDIBLE 0.02 | chicken or meat dish | rotten decaying food
f15.jpg | NON-EDIBLE 0.24 | bread or bun | moldy spoiled food


In [12]:
NGOS = [
  {"name": "Annapoorna Trust", "lat": 10.38, "lng": 78.82, "capacity": 150, "radius": 15,
   "text": "Serves hot cooked meals like rice, sambar, curry and chicken to homeless people and daily wage workers"},
  {"name": "Little Hands Orphanage", "lat": 10.40, "lng": 78.80, "capacity": 60, "radius": 12,
   "text": "Children's home that accepts bread, bakery items, fruits, biscuits, sweets and milk"},
  {"name": "Green Basket Foundation", "lat": 10.35, "lng": 78.85, "capacity": 200, "radius": 20,
   "text": "Distributes fresh vegetables and fruits to low-income families"},
  {"name": "Sunset Elders Home", "lat": 10.42, "lng": 78.78, "capacity": 40, "radius": 10,
   "text": "Old age home that needs soft cooked food like rice, soup, idli, fruits and milk"},
  {"name": "Hope Night Shelter", "lat": 10.37, "lng": 78.83, "capacity": 120, "radius": 15,
   "text": "Night shelter providing dinner such as cooked rice, noodles, curry and bread to homeless people"},
  {"name": "Bright Future School Meals", "lat": 10.39, "lng": 78.86, "capacity": 250, "radius": 18,
   "text": "School meal program for children that accepts cooked meals, packaged snacks, biscuits and fruits"},
]

RECYCLERS = [
  {"name": "Gokul Goshala", "stream": "animal", "lat": 10.36, "lng": 78.84, "radius": 25,
   "text": "Cattle shelter that accepts stale bread, vegetable peels, fruit scraps and dry leftovers as animal feed"},
  {"name": "Happy Paws Shelter", "stream": "animal", "lat": 10.41, "lng": 78.81, "radius": 15,
   "text": "Dog and cat shelter that accepts leftover rice, bread and cooked scraps without mold"},
  {"name": "Green Earth Composters", "stream": "compost", "lat": 10.34, "lng": 78.87, "radius": 30,
   "text": "Composting unit that turns spoiled food, rotten fruits, vegetable waste and peels into organic compost"},
  {"name": "Vermi Farm Co-op", "stream": "compost", "lat": 10.43, "lng": 78.79, "radius": 25,
   "text": "Vermicompost farm that accepts rotten vegetables, fruit waste and spoiled food"},
  {"name": "BioGas Energy Plant", "stream": "biogas", "lat": 10.33, "lng": 78.88, "radius": 40,
   "text": "Biogas plant converting food waste, bones, meat waste and cooked leftovers into biogas fuel"},
  {"name": "Rural Biogas Unit", "stream": "biogas", "lat": 10.45, "lng": 78.75, "radius": 30,
   "text": "Community biogas digester that accepts kitchen waste, bones and meat scraps to produce cooking gas"},
]

def build_index(texts):
    emb = mini.encode(texts, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])     # inner product = cosine similarity
    index.add(emb)
    return index

# FAISS index for NGOs
ngo_index = build_index([n["text"] for n in NGOS])

# One FAISS index per recovery stream
REC_INDEX = {}
for stream in ["animal", "compost", "biogas"]:
    items = [r for r in RECYCLERS if r["stream"] == stream]
    REC_INDEX[stream] = (build_index([r["text"] for r in items]), items)

print(f"{len(NGOS)} NGOs and {len(RECYCLERS)} recyclers indexed in FAISS")

6 NGOs and 6 recyclers indexed in FAISS


In [13]:
def rank_ngos(query, quantity, lat, lng, top_k=3):
    q = mini.encode([query], normalize_embeddings=True).astype("float32")
    sims, ids = ngo_index.search(q, len(NGOS))
    out = []
    for sim, i in zip(sims[0], ids[0]):
        n = NGOS[i]
        dist = haversine(lat, lng, n["lat"], n["lng"])
        if dist > n["radius"]:
            continue                                   # outside service area
        sim = max(0.0, float(sim))
        prox = max(0.0, 1 - dist / n["radius"])
        cap = min(1.0, n["capacity"] / max(quantity, 1))
        score = 0.5 * sim + 0.3 * prox + 0.2 * cap
        out.append({"name": n["name"], "score": round(score * 100, 1),
                    "reason": f"{sim:.0%} food match, {dist:.1f} km away, capacity {n['capacity']} meals"})
    return sorted(out, key=lambda x: x["score"], reverse=True)[:top_k]

def rank_recyclers(stream, query, lat, lng, top_k=2):
    index, items = REC_INDEX[stream]
    q = mini.encode([query], normalize_embeddings=True).astype("float32")
    sims, ids = index.search(q, len(items))
    out = []
    for sim, i in zip(sims[0], ids[0]):
        r = items[i]
        dist = haversine(lat, lng, r["lat"], r["lng"])
        if dist > r["radius"]:
            continue
        prox = max(0.0, 1 - dist / r["radius"])
        score = 0.6 * max(0.0, float(sim)) + 0.4 * prox
        out.append({"name": r["name"], "score": round(score * 100, 1),
                    "reason": f"{max(0.0, float(sim)):.0%} waste match, {dist:.1f} km away"})
    return sorted(out, key=lambda x: x["score"], reverse=True)[:top_k]

In [14]:
PATHWAYS = {
    "human":   "Food rescue → NGO (human consumption)",
    "animal":  "Animal feed → goshala / animal shelter",
    "compost": "Bio-compost → composting / vermicompost unit",
    "biogas":  "Bio-energy → biogas plant",
}

IMPACT = []     # every processed donation is logged here

def dashboard():
    if not IMPACT:
        print("No donations processed yet"); return
    df = pd.DataFrame(IMPACT)
    print("=== SURPLUSLINK IMPACT ===")
    print("Donations processed :", len(df))
    print("Meals rescued       :", int(df.loc[df.stream == "human", "meals"].sum()))
    for s, label in [("animal", "Animal feed (kg)"), ("compost", "Compost (kg)"), ("biogas", "Biogas feedstock (kg)")]:
        print(f"{label:<20}:", round(df.loc[df.stream == s, "weight_kg"].sum(), 1))
    print("Waste diverted from landfill (kg):",
          round(df.loc[df.stream != 'human', 'weight_kg'].sum(), 1))
    print("Needs manual review :", int(df.needs_review.sum()))
    return df

In [15]:
def process_donation(path, quantity=50, weight_kg=10, hours_since_cooked=1,
                     storage="room", lat=10.38, lng=78.82, log=True):
    cap = caption(path)
    c = classify_donation(path)
    reason = ""

    # ---- Decide the stream ----
    if c["edible"]:
        safe, why = safety_check(c["food_category"], hours_since_cooked, storage)
        if safe:
            stream = "human"
        else:
            stream, reason = "compost", f"Unsafe: {why}"
    else:
        stream = c["non_edible_stream"]
        reason = f"Detected as {c['non_edible_label']}"

    # ---- Match partners with FAISS ----
    if stream == "human":
        matches = rank_ngos(f"{c['food_type']} ({c['food_category']}): {cap}", quantity, lat, lng)
    else:
        matches = rank_recyclers(stream, f"{c['non_edible_label']}: {cap}", lat, lng)

    result = {
        "image": os.path.basename(path), "caption": cap,
        "food": f"{c['food_type']} ({c['food_category']})",
        "edible": c["edible"], "edible_score": c["edible_score"],
        "stream": stream, "pathway": PATHWAYS[stream], "reason": reason,
        "needs_review": c["needs_review"], "matches": matches,
    }

    if log:
        IMPACT.append({"image": result["image"], "stream": stream,
                       "meals": quantity if stream == "human" else 0,
                       "weight_kg": weight_kg, "needs_review": c["needs_review"]})
    return result

def show(r):
    print("\n" + "=" * 62)
    print(f"{r['image']} | {r['caption']}")
    print(f"Food: {r['food']} | edible score: {r['edible_score']}"
          + ("  ⚠ REVIEW" if r["needs_review"] else ""))
    print(f"Pathway: {r['pathway']}" + (f"  ({r['reason']})" if r["reason"] else ""))
    if not r["matches"]:
        print("  - No partner within range")
    for m in r["matches"]:
        print(f"  - {m['name']}  [{m['score']}]  {m['reason']}")

In [24]:
IMPACT.clear()
rows = []
for f in find_images():
    r = process_donation(os.path.join(IMAGE_DIR, f), quantity=50, weight_kg=10,
                         hours_since_cooked=1, storage="room")
    show(r)
    rows.append({"image": r["image"], "caption": r["caption"], "food": r["food"],
                 "edible_score": r["edible_score"], "stream": r["stream"],
                 "top_match": r["matches"][0]["name"] if r["matches"] else "none",
                 "review": r["needs_review"], "true_food": "", "correct": ""})

results = pd.DataFrame(rows)
results.to_csv("/vision_results.csv", index=False)
results


f1.jpg | a bowl of food with meat and vegetables in it
Food: curry or gravy (cooked meals) | edible score: 0.66
Pathway: Food rescue → NGO (human consumption)
  - Annapoorna Trust  [78.6]  57% food match, 0.0 km away, capacity 150 meals
  - Hope Night Shelter  [67.3]  41% food match, 1.6 km away, capacity 120 meals
  - Bright Future School Meals  [61.3]  38% food match, 4.5 km away, capacity 250 meals

f2.jpeg | a glass bowl filled with meat and cheese
Food: pizza (cooked meals) | edible score: 0.84
Pathway: Food rescue → NGO (human consumption)
  - Annapoorna Trust  [73.0]  46% food match, 0.0 km away, capacity 150 meals
  - Hope Night Shelter  [66.7]  40% food match, 1.6 km away, capacity 120 meals
  - Bright Future School Meals  [62.9]  41% food match, 4.5 km away, capacity 250 meals

f3.jpg | a pan full of food sitting on a stove
Food: chicken or meat dish (cooked meals) | edible score: 0.96
Pathway: Food rescue → NGO (human consumption)
  - Annapoorna Trust  [76.7]  53% food matc

,image,caption,food,edible_score,stream,top_match,review,true_food,correct
0,f1.jpg,a bowl of food with meat and vegetables in it,curry or gravy (cooked meals),0.66,human,Annapoorna Trust,False,,
1,f2.jpeg,a glass bowl filled with meat and cheese,pizza (cooked meals),0.84,human,Annapoorna Trust,False,,
2,f3.jpg,a pan full of food sitting on a stove,chicken or meat dish (cooked meals),0.96,human,Annapoorna Trust,False,,
3,f4.jpg,a bowl of food sitting on top of a table,rice dish or biryani (cooked meals),0.98,human,Annapoorna Trust,False,,
4,f5.webp,a plate with a pizza on top of it,pizza (cooked meals),0.99,human,Annapoorna Trust,False,,
5,f6.jpg,a pot filled with soup sitting on top of a woo...,curry or gravy (cooked meals),0.93,human,Annapoorna Trust,False,,
6,f7.jpeg,a plate of noodles with meat and vegetables,noodles or pasta (cooked meals),0.83,human,Annapoorna Trust,False,,
7,f8.webp,a white bowl filled with food on top of a wood...,curry or gravy (cooked meals),0.99,human,Annapoorna Trust,False,,
8,f9.jpeg,a silver bowl filled with rice and vegetables,rice dish or biryani (cooked meals),0.88,human,Annapoorna Trust,False,,
9,f10.jpg,a skill skill skill skill skill skill skill sk...,rice dish or biryani (cooked meals),0.99,human,Annapoorna Trust,False,,
